<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026psy3a_Dell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations
from dataclasses import dataclass, asdict
import numpy as np
from typing import Dict, List, Tuple, Optional, Literal

JoltMode = Literal["lex", "phon", "both", "none"]

# -----------------------------
# Data structures
# -----------------------------
@dataclass(frozen=True)
class LexItem:
    word: str
    sem: np.ndarray              # binary features shape (S,)
    phon: Tuple[int, ...]        # phoneme sequence (p1,p2,...)

@dataclass(frozen=True)
class ObservedStats:
    """
    What you can realistically get from picture naming alone:
    - proportions of error categories
    Optionally: target-specific confusion matrix if you have it.
    """
    rates: Dict[str, float]                  # keys: correct/semantic/phonological/mixed/unrelated/omission
    # Optional richer observation (still from naming):
    # confusion[target, response] counts
    confusion: Optional[np.ndarray] = None   # shape (W, W)
    n_trials: Optional[int] = None           # total trials behind rates/confusion

@dataclass(frozen=True)
class Params:
    # Connection strengths (impairment knobs)
    w_sem_lex: float = 1.0
    w_lex_sem: float = 1.0
    w_lex_phon: float = 1.0
    w_phon_lex: float = 1.0

    # Dynamics
    decay: float = 0.25
    gain: float = 1.0
    noise_sd: float = 0.05          # core impairment axis

    # Timing
    n_steps: int = 16
    jolt_step: int = 8              # 1..n_steps (Dell-ish timing)

    # Jolt
    jolt_mode: JoltMode = "lex"
    jolt_lex: float = 0.6
    jolt_phon: float = 0.0

    # Selection stochasticity (separate from dynamic noise)
    temperature: float = 0.7

    # Omission rule (very important in aphasia fits)
    # if max lexical activation < threshold => omission
    omission_thresh: float = 0.05

# -----------------------------
# Core model: Interactive Activation with Jolt
# -----------------------------
class FoygelDell2000Like:
    """
    Minimal-but-useful faithful core:
      Sem <-> Lex <-> Phon
    with:
      - 16 iterations
      - jolt at step 8
      - noise & connection strength impairment
      - output: a RESPONSE WORD (like picture naming)
    Notes:
      This is deliberately structured to highlight IDENTIFIABILITY problems.
    """

    def __init__(self, items: List[LexItem], n_phonemes: int, seed: int = 0):
        assert len(items) >= 2
        self.items = items
        self.W = len(items)
        self.S = int(items[0].sem.shape[0])
        self.P = int(n_phonemes)
        self.rng = np.random.default_rng(seed)

        self.word2idx = {it.word: i for i, it in enumerate(items)}

        # Sem->Lex (W,S), Lex->Sem (S,W)
        self.M_sem_lex = np.stack([it.sem for it in items], axis=0).astype(float)
        self.M_lex_sem = self.M_sem_lex.T.copy()

        # Lex->Phon (P,W) as normalized bag-of-phonemes (keeps code simple)
        self.M_lex_phon = np.zeros((self.P, self.W), dtype=float)
        for w, it in enumerate(items):
            for p in it.phon:
                self.M_lex_phon[p, w] += 1.0
        col = np.linalg.norm(self.M_lex_phon, axis=0, keepdims=True) + 1e-8
        self.M_lex_phon /= col

        # Phon->Lex (W,P)
        self.M_phon_lex = self.M_lex_phon.T.copy()

    # ----- dynamics -----
    def _step(self, a: np.ndarray, net_in: np.ndarray, decay: float, noise_sd: float) -> np.ndarray:
        noise = self.rng.normal(0.0, noise_sd, size=a.shape)
        a_new = (1 - decay) * a + net_in + noise
        return np.clip(a_new, 0.0, None)

    def _softmax(self, x: np.ndarray, temperature: float) -> np.ndarray:
        t = max(temperature, 1e-8)
        z = (x / t) - np.max(x / t)
        ez = np.exp(z)
        return ez / np.sum(ez)

    # ----- error classification (from naming only) -----
    @staticmethod
    def _jaccard(a: np.ndarray, b: np.ndarray) -> float:
        inter = np.logical_and(a > 0, b > 0).sum()
        union = np.logical_or(a > 0, b > 0).sum()
        return float(inter / union) if union > 0 else 0.0

    def classify(self, tidx: int, ridx: int, sem_thr=0.35, phon_thr=0.35) -> str:
        if ridx < 0:
            return "omission"
        if tidx == ridx:
            return "correct"

        t = self.items[tidx]
        r = self.items[ridx]

        sem_sim = self._jaccard(t.sem, r.sem)
        tset, rset = set(t.phon), set(r.phon)
        phon_sim = len(tset & rset) / max(len(tset | rset), 1)

        sem_related = sem_sim >= sem_thr
        phon_related = phon_sim >= phon_thr

        if sem_related and phon_related:
            return "mixed"
        if sem_related:
            return "semantic"
        if phon_related:
            return "phonological"
        return "unrelated"

    # ----- one trial -----
    def run_trial(self, target_word: str, p: Params) -> int:
        tidx = self.word2idx[target_word]
        target = self.items[tidx]

        # activations
        a_sem = target.sem.astype(float).copy()  # clamped input
        a_lex = np.zeros(self.W, dtype=float)
        a_phon = np.zeros(self.P, dtype=float)

        for step in range(1, p.n_steps + 1):
            # compute net inputs
            net_lex = p.gain * (
                p.w_sem_lex * (self.M_sem_lex @ a_sem) +
                p.w_phon_lex * (self.M_phon_lex @ a_phon)
            )
            net_phon = p.gain * (p.w_lex_phon * (self.M_lex_phon @ a_lex))

            # JOLT (pulse-like external input)
            if step == p.jolt_step and p.jolt_mode != "none":
                if p.jolt_mode in ("lex", "both"):
                    net_lex = net_lex.copy()
                    net_lex[tidx] += p.jolt_lex
                if p.jolt_mode in ("phon", "both"):
                    net_phon = net_phon.copy()
                    net_phon += p.jolt_phon * self.M_lex_phon[:, tidx]

            # update
            a_lex = self._step(a_lex, net_lex, p.decay, p.noise_sd)
            a_phon = self._step(a_phon, net_phon, p.decay, p.noise_sd)

            # optional weak lex->sem feedback (kept small)
            net_sem = p.gain * (p.w_lex_sem * (self.M_lex_sem @ a_lex))
            a_sem = np.clip(target.sem.astype(float) + 0.1 * net_sem, 0.0, None)

        # omission rule (common in aphasia modeling)
        if float(a_lex.max()) < p.omission_thresh:
            return -1

        # select response word
        probs = self._softmax(a_lex, p.temperature)
        ridx = int(self.rng.choice(self.W, p=probs))
        return ridx

    # ----- simulation for naming task -----
    def simulate_naming(
        self,
        targets: List[str],
        p: Params,
        n_rep: int,
        seed: Optional[int] = None,
        return_confusion: bool = True
    ) -> Tuple[Dict[str, float], Optional[np.ndarray]]:
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        counts: Dict[str, int] = {k: 0 for k in ["correct","semantic","phonological","mixed","unrelated","omission"]}
        conf = np.zeros((self.W, self.W), dtype=int) if return_confusion else None

        for _ in range(n_rep):
            for tw in targets:
                tidx = self.word2idx[tw]
                ridx = self.run_trial(tw, p)
                et = self.classify(tidx, ridx)
                counts[et] += 1
                if conf is not None and ridx >= 0:
                    conf[tidx, ridx] += 1

        total = sum(counts.values())
        rates = {k: v/total for k, v in counts.items()}
        return rates, conf

# -----------------------------
# Fitting objective: deliberately "naming-only"
# -----------------------------
def distance_on_rates(sim_rates: Dict[str, float], obs_rates: Dict[str, float], eps: float = 1e-12) -> float:
    """
    Simple distance: sum of squared errors on category rates.
    (You could also use KL divergence; SSE is easier to interpret)
    """
    keys = sorted(set(sim_rates.keys()) | set(obs_rates.keys()))
    d = 0.0
    for k in keys:
        a = float(sim_rates.get(k, 0.0))
        b = float(obs_rates.get(k, 0.0))
        d += (a - b) ** 2
    return d

def multinomial_nll_from_rates(sim_rates: Dict[str, float], obs_counts: Dict[str, int], eps: float = 1e-12) -> float:
    """
    If you have counts (not just proportions), you can compute a multinomial NLL.
    This is more 'statistically honest' and makes identifiability issues even clearer.
    """
    nll = 0.0
    for k, c in obs_counts.items():
        pk = float(sim_rates.get(k, 0.0))
        pk = max(pk, eps)
        nll += -c * np.log(pk)
    return float(nll)

# -----------------------------
# Grid search designed to EXPOSE non-identifiability
# -----------------------------
@dataclass(frozen=True)
class FitResult:
    params: Params
    score: float
    sim_rates: Dict[str, float]

def grid_search(
    model: FoygelDell2000Like,
    targets: List[str],
    obs: ObservedStats,
    base: Params,
    grid: Dict[str, np.ndarray],
    n_rep: int = 200,
    top_n: int = 25,
    seed0: int = 1234,
) -> List[FitResult]:
    """
    grid keys can include any Params field, e.g.
      - noise_sd
      - w_sem_lex
      - w_lex_phon
      - w_phon_lex
      - jolt_mode (handled separately below)
      - jolt_lex, jolt_phon
    This returns MANY near-optimal fits to make "cause not identifiable" explicit.
    """
    # Prepare observed: if only proportions exist, use rate-distance
    use_nll = (obs.n_trials is not None) and (obs.confusion is None) and (obs.rates is not None)
    # If user provides counts separately, adapt; here assume proportions -> SSE by default.

    # Expand jolt_mode if provided as list
    jolt_modes: List[JoltMode]
    if "jolt_mode" in grid:
        jolt_modes = list(grid["jolt_mode"])  # type: ignore
    else:
        jolt_modes = [base.jolt_mode]

    # Build a list of parameter names excluding jolt_mode
    param_names = [k for k in grid.keys() if k != "jolt_mode"]
    axes = [grid[k] for k in param_names]
    mesh = np.array(np.meshgrid(*axes, indexing="ij"), dtype=object)
    combos = mesh.reshape(len(param_names), -1).T  # (N, D)

    results: List[FitResult] = []
    combo_id = 0

    for jm in jolt_modes:
        for row in combos:
            kwargs = asdict(base)
            kwargs["jolt_mode"] = jm
            for k, v in zip(param_names, row):
                kwargs[k] = float(v) if isinstance(v, (int, float, np.floating)) else v
            p = Params(**kwargs)

            sim_rates, _ = model.simulate_naming(
                targets=targets,
                p=p,
                n_rep=n_rep,
                seed=seed0 + combo_id,
                return_confusion=False
            )
            combo_id += 1

            score = distance_on_rates(sim_rates, obs.rates)

            results.append(FitResult(params=p, score=float(score), sim_rates=sim_rates))

    # Sort and keep top_n (but not just best-1)
    results.sort(key=lambda r: r.score)
    return results[:top_n]

# -----------------------------
# Interpretation helpers (make the identifiability issue obvious)
# -----------------------------
def summarize_fitset(fits: List[FitResult]) -> Dict[str, Tuple[float, float]]:
    """
    For each parameter, report (min, max) across near-optimal fits.
    If these ranges are wide, naming-only is not identifying them.
    """
    if not fits:
        return {}
    pdicts = [asdict(fr.params) for fr in fits]
    keys = pdicts[0].keys()

    summary: Dict[str, Tuple[float, float]] = {}
    for k in keys:
        # skip non-numeric except jolt_mode
        if k == "jolt_mode":
            continue
        vals = [p[k] for p in pdicts]
        if all(isinstance(x, (int, float, np.floating)) for x in vals):
            summary[k] = (float(np.min(vals)), float(np.max(vals)))
    return summary

# -----------------------------
# Toy lexicon (replace with your real one)
# -----------------------------
def make_toy_lexicon() -> Tuple[List[LexItem], int]:
    P = 12
    def sem_vec(*idxs: int) -> np.ndarray:
        v = np.zeros(10, dtype=int)
        v[list(idxs)] = 1
        return v
    items = [
        LexItem("cat", sem_vec(0,1,2), (0,1,2)),
        LexItem("dog", sem_vec(0,1,3), (0,4,2)),
        LexItem("rat", sem_vec(0,2,4), (5,1,2)),
        LexItem("car", sem_vec(5,6),   (5,7,8)),
        LexItem("bus", sem_vec(5,7),   (9,7,10)),
        LexItem("cup", sem_vec(8,9),   (0,7,11)),
    ]
    return items, P

In [2]:
items, P = make_toy_lexicon()
model = FoygelDell2000Like(items, n_phonemes=P, seed=0)
targets = [it.word for it in items]

# Suppose "observed" from a patient: (dummy numbers)
obs = ObservedStats(
    rates={
        "correct": 0.55,
        "semantic": 0.12,
        "phonological": 0.12,
        "mixed": 0.06,
        "unrelated": 0.10,
        "omission": 0.05,
    },
    n_trials=None)

base = Params(
    n_steps=16, jolt_step=8,
    jolt_mode="lex",
    # start values (won't matter much once we grid-search)
    w_sem_lex=1.2, w_lex_phon=1.0, w_phon_lex=0.6,
    noise_sd=0.06,
    jolt_lex=0.4, jolt_phon=0.4,
    temperature=0.7,
    omission_thresh=0.05
)

grid = {
    "jolt_mode": np.array(["lex", "phon", "both"], dtype=object),
    "noise_sd": np.linspace(0.02, 0.12, 6),
    "w_sem_lex": np.linspace(0.6, 1.4, 5),
    "w_lex_phon": np.linspace(0.6, 1.4, 5),
    "w_phon_lex": np.linspace(0.2, 1.0, 5),
    "jolt_lex": np.linspace(0.0, 1.0, 6),
    "jolt_phon": np.linspace(0.0, 1.0, 6),
}

fits = grid_search(model, targets, obs, base, grid, n_rep=2, top_n=2, seed0=777)
#fits = grid_search(model, targets, obs, base, grid, n_rep=150, top_n=20, seed0=777)
print("Top fits (score, jolt_mode, noise, w_sem_lex, w_lex_phon, w_phon_lex, jlex, jphon):")
for fr in fits[:10]:
    p = fr.params
    print(fr.score, p.jolt_mode, p.noise_sd,
          p.w_sem_lex, p.w_lex_phon, p.w_phon_lex,
          p.jolt_lex, p.jolt_phon, fr.sim_rates)

print("\nParameter ranges across top fits (non-identifiability diagnostic):")
print(summarize_fitset(fits))

Top fits (score, jolt_mode, noise, w_sem_lex, w_lex_phon, w_phon_lex, jlex, jphon):
0.11295555555555555 lex 0.02 0.6 0.6 0.2 0.0 0.0 {'correct': 0.5, 'semantic': 0.0, 'phonological': 0.0, 'mixed': 0.3333333333333333, 'unrelated': 0.16666666666666666, 'omission': 0.0}
0.11295555555555555 lex 0.02 0.6 0.6 0.2 0.0 0.2 {'correct': 0.5, 'semantic': 0.0, 'phonological': 0.0, 'mixed': 0.3333333333333333, 'unrelated': 0.16666666666666666, 'omission': 0.0}

Parameter ranges across top fits (non-identifiability diagnostic):
{'w_sem_lex': (0.6, 0.6), 'w_lex_sem': (1.0, 1.0), 'w_lex_phon': (0.6, 0.6), 'w_phon_lex': (0.2, 0.2), 'decay': (0.25, 0.25), 'gain': (1.0, 1.0), 'noise_sd': (0.02, 0.02), 'n_steps': (16.0, 16.0), 'jolt_step': (8.0, 8.0), 'jolt_lex': (0.0, 0.0), 'jolt_phon': (0.0, 0.2), 'temperature': (0.7, 0.7), 'omission_thresh': (0.05, 0.05)}
